# Toy example: 2D damped linear oscillator

**Purpose:** verification example with a known exact analytical solution.

The true governing equations are:
$$\dot{x} = \mu x + \omega y, \qquad \dot{y} = -\omega x + \mu y$$
with $\mu = -0.1$ (decay rate) and $\omega = 2.0$ (oscillation frequency).

Starting from $(x_0, 0)$ the exact analytical solution is:
$$x(t) = x_0 \, e^{\mu t} \cos(\omega t), \qquad y(t) = -x_0 \, e^{\mu t} \sin(\omega t)$$

SINDy is given a **degree-2 polynomial library** (all monomials up to $x^2, xy, y^2$). The sparsity
penalty should zero every nonlinear feature and recover exactly the four linear coefficients
$\mu, \omega, -\omega, \mu$. We verify this by comparing the discovered coefficients to the
analytical ground truth.

In [ ]:
import sys
from pathlib import Path

root = Path(".").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import os
import numpy as np
import matplotlib.pyplot as plt

from damped_oscillator.simulate import simulate_damped_oscillator, exact_solution
from damped_oscillator.model import build_oscillator_model, oscillator_sindy_config
from damped_oscillator.integrated import integrate_discovered_2d
from damped_oscillator.plot import plot_phase_portrait, plot_time_series
from sindy.pipeline import run_sindy_pipeline_general
from idtools.compare_to_truth import compare_to_truth

%matplotlib inline

## 1. Simulate and compare to analytical solution

In [ ]:
MU, OMEGA = -0.1, 2.0
X0 = (2.0, 0.0)

t, Z_phys = simulate_damped_oscillator(
    t_span=(0.0, 30.0),
    n_points=3000,
    x0=X0,
    mu=MU,
    omega=OMEGA,
)
Z_exact = exact_solution(t, x0=X0[0], mu=MU, omega=OMEGA)

max_err = float(np.max(np.abs(Z_phys - Z_exact)))
print(f"Numerical vs analytical max |error|: {max_err:.2e}  (should be ~1e-9 or smaller)")
print(f"Data shape: {Z_phys.shape},  t range: [{t[0]:.1f}, {t[-1]:.1f}]")

## 2. Model, polynomial budget, and exact derivatives

In [ ]:
model = build_oscillator_model(mu=MU, omega=OMEGA)

# Polynomial-only budget (no trig / inv / sat terms)
budget = {
    "x": {"lin": 2, "trig": 0, "inv": 0, "sat": 0},
    "y": {"lin": 2, "trig": 0, "inv": 0, "sat": 0},
}

# Exact derivatives from the known RHS (Brunton-style clean-derivative validation)
Z_dot_phys = np.column_stack([
    model.rhs_lambdified[i](*Z_phys.T) for i in range(len(model.rhs_lambdified))
])

print("True equations:")
for name, expr in zip(model.target_names, model.rhs_symbolic):
    print(f"  d{name}/dt = {expr}")

## 3. SINDy fit (Pareto + BIC, degree-2 library)

The library contains: $1, x, y, x^2, xy, y^2$ — six features per equation. The true model only
uses the two linear terms in each equation, so correct sparse recovery means four nonzero
coefficients out of twelve total.

In [ ]:
output_dir = "outputs/damped_oscillator/"
os.makedirs(output_dir, exist_ok=True)

cfg = oscillator_sindy_config()
res = run_sindy_pipeline_general(
    t=t,
    Z_phys=Z_phys,
    model=model,
    budget=budget,
    config=cfg,
    Z_dot_phys=Z_dot_phys,
    run_diagnostics=True,
    output_dir=output_dir,
)

val = res["validation"]
print(f"R²(mean): {val['r2_mean']:.6f},  RMSE: {val['rmse']:.4g}")
r2b = val.get("r2_by_state")
if r2b is not None:
    r2b = np.asarray(r2b).ravel()
    print(f"R² per equation:  dx={r2b[0]:.6f},  dy={r2b[1]:.6f}")
print("\nDiscovered equations:")
for name, expr in res["fit"]["equations"].items():
    print(f"  d{name}/dt = {expr}")

## 4. Coefficient verification against analytical ground truth

`compare_to_truth` projects the symbolic true equations onto the same library and reports
coefficient-level relative errors. For a clean linear system with exact derivatives we expect
near-machine-precision recovery.

In [ ]:
truth = compare_to_truth(res, verbose=True)

## 4b. BIC selection visualised

The top panel shows the NMSE vs complexity Pareto frontier; the bottom shows the BIC score
for every threshold sweep point. The red star marks the model selected by minimum BIC — the
balance between fit quality ($n \ln \text{MSE}$) and model complexity ($k \ln n$).

In [ ]:
from sindy.pareto import plot_bic_selection

fig, _ = plot_bic_selection(
    res["fit"],
    title="Damped oscillator — BIC selection",
    out_path=output_dir + "bic_selection.png",
)
plt.show()

### Coefficient tables

Side-by-side: true $\Xi$ vs discovered $\Xi$ and relative errors.

In [ ]:
from sindy.diagnostics_plots import plot_xi_heatmaps

fit = res["fit"]
target_names = fit["target_names"]
kept_names   = fit["kept_names"]

xi_true = truth["xi_true"]
xi_disc = truth["xi_disc"]

plot_xi_heatmaps(
    kept_names=kept_names[:xi_true.shape[0]],
    Xi_true=xi_true,
    Xi_disc=xi_disc,
    target_names=target_names,
    output_dir=output_dir,
    prefix="structure_verification",
)
print(f"Coefficient heatmaps saved to {output_dir}")

## 5. Forward integration and phase portrait

Integrate the discovered ODE from the same initial condition and compare the trajectory to the
reference simulation and the analytical solution.

In [ ]:
Z_sindy = integrate_discovered_2d(res["fit"], z0=Z_phys[0], t_eval=t)

rmse = float(np.sqrt(np.mean((Z_phys - Z_sindy) ** 2)))
print(f"RMSE(state, SINDy vs reference): {rmse:.4g}")

fig, ax = plot_phase_portrait(
    Z_phys,
    Z_sindy,
    Z_exact,
    out_path=output_dir + "phase_portrait.png",
)
plt.show()

In [ ]:
fig, axes = plot_time_series(
    t,
    Z_phys,
    Z_sindy,
    Z_exact,
    out_path=output_dir + "time_series.png",
)
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| True equations | $\dot x = -0.1x + 2y$, $\dot y = -2x - 0.1y$ |
| Library size | 6 features × 2 equations = 12 coefficients |
| Nonzero in truth | 4 |
| Coefficient correlation | see `compare_to_truth` output |
| Max relative error on nonzero terms | see `compare_to_truth` output |

The analytical solution $x(t) = 2e^{-0.1t}\cos(2t)$ provides exact ground truth at every time
step. Agreement between the numerical integration, the discovered model, and the analytical
formula confirms both the simulator and the SINDy recovery are correct.